In [2]:
import os
os.chdir(r'C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11')

In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [4]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [5]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [6]:
model1 = MLP()
model1.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 504)               0         
                                                                 
 dense (Dense)               (None, 32)                16160     
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 16193 (63.25 KB)
Trainable params: 16193 (63.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [7]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


In [14]:
checkpoints = r'C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [15]:
os.path.exists(JSON_PATH)

False

In [16]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [17]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading <keras.src.engine.sequential.Sequential object at 0x00000298BA756260>...


OSError: Unable to load model. Filepath is not an hdf5 file (or h5py is not available) or SavedModel. Received: filepath=<keras.src.engine.sequential.Sequential object at 0x00000298BA756260>

In [18]:
import os
path_dataset =r'C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\Muhib Ur-Rahman\anaconda3\envs\dsp\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [19]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.006197214126586914 sec


In [20]:
train_X.shape

(835, 24, 21)

In [21]:
epochs = 60
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/60


27/27 [==============================] - ETA: 0s - loss: 0.1545 - mae: 0.1545 - mape: 77.0367
Epoch 1: val_loss improved from inf to 0.06953, saving model to C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\E1-cp-0001-loss0.07.h5
27/27 [==============================] - 4s 37ms/step - loss: 0.1545 - mae: 0.1545 - mape: 77.0367 - val_loss: 0.0695 - val_mae: 0.0695 - val_mape: 20.7884
Epoch 2/60
21/27 [======================>.......] - ETA: 0s - loss: 0.0762 - mae: 0.0762 - mape: 51.7979

c:\Users\Muhib Ur-Rahman\anaconda3\envs\dsp\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(



Epoch 2: val_loss did not improve from 0.06953
27/27 [==============================] - 2s 62ms/step - loss: 0.0746 - mae: 0.0746 - mape: 47.8422 - val_loss: 0.0821 - val_mae: 0.0821 - val_mape: 24.7472
Epoch 3/60
24/27 [=========================>....] - ETA: 0s - loss: 0.0626 - mae: 0.0626 - mape: 39.1417
Epoch 3: val_loss did not improve from 0.06953
27/27 [==============================] - 1s 29ms/step - loss: 0.0621 - mae: 0.0621 - mape: 37.5617 - val_loss: 0.0871 - val_mae: 0.0871 - val_mape: 27.7562
Epoch 4/60
22/27 [=======================>......] - ETA: 0s - loss: 0.0716 - mae: 0.0716 - mape: 35.1455
Epoch 4: val_loss improved from 0.06953 to 0.05393, saving model to C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\E1-cp-0004-loss0.05.h5


c:\Users\Muhib Ur-Rahman\anaconda3\envs\dsp\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 51ms/step - loss: 0.0704 - mae: 0.0704 - mape: 33.3873 - val_loss: 0.0539 - val_mae: 0.0539 - val_mape: 16.3041
Epoch 5/60
21/27 [======================>.......] - ETA: 0s - loss: 0.0626 - mae: 0.0626 - mape: 33.7197
Epoch 5: val_loss improved from 0.05393 to 0.05123, saving model to C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\E1-cp-0005-loss0.05.h5


...

27/27 [==============================] - 1s 40ms/step - loss: 0.0333 - mae: 0.0333 - mape: 18.0028 - val_loss: 0.0309 - val_mae: 0.0309 - val_mape: 10.0943
Epoch 32/60
26/27 [===========================>..] - ETA: 0s - loss: 0.0329 - mae: 0.0329 - mape: 17.0276
Epoch 32: val_loss did not improve from 0.03094
27/27 [==============================] - 1s 49ms/step - loss: 0.0329 - mae: 0.0329 - mape: 16.9881 - val_loss: 0.0312 - val_mae: 0.0312 - val_mape: 9.8532
Epoch 33/60
23/27 [========================>.....] - ETA: 0s - loss: 0.0288 - mae: 0.0288 - mape: 14.2083
Epoch 33: val_loss did not improve from 0.03094
27/27 [==============================] - 2s 71ms/step - loss: 0.0290 - mae: 0.0290 - mape: 13.8725 - val_loss: 0.0356 - val_mae: 0.0356 - val_mape: 11.6258
Epoch 34/60
26/27 [===========================>..] - ETA: 0s - loss: 0.0300 - mae: 0.0300 - mape: 13.7932
Epoch 34: val_loss did not improve from 0.03094
27/27 [==============================] - 1s 28ms/step - loss: 0.0300 - 

c:\Users\Muhib Ur-Rahman\anaconda3\envs\dsp\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 54ms/step - loss: 0.0296 - mae: 0.0296 - mape: 14.1444 - val_loss: 0.0288 - val_mae: 0.0288 - val_mape: 9.0202
Epoch 36/60
20/27 [=====================>........] - ETA: 0s - loss: 0.0259 - mae: 0.0259 - mape: 12.1671
Epoch 36: val_loss did not improve from 0.02878
27/27 [==============================] - 1s 25ms/step - loss: 0.0254 - mae: 0.0254 - mape: 11.3907 - val_loss: 0.0353 - val_mae: 0.0353 - val_mape: 11.6714
Epoch 37/60
22/27 [=======================>......] - ETA: 0s - loss: 0.0247 - mae: 0.0247 - mape: 11.3742
Epoch 37: val_loss did not improve from 0.02878
27/27 [==============================] - 1s 25ms/step - loss: 0.0250 - mae: 0.0250 - mape: 11.2218 - val_loss: 0.0292 - val_mae: 0.0292 - val_mape: 9.5222
Epoch 38/60
17/27 [=================>............] - ETA: 0s - loss: 0.0306 - mae: 0.0306 - mape: 14.2300
Epoch 38: val_loss did not improve from 0.02878
27/27 [==============================] - 1s 21ms/step - loss: 0.0296 - m

c:\Users\Muhib Ur-Rahman\anaconda3\envs\dsp\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 2s 75ms/step - loss: 0.0248 - mae: 0.0248 - mape: 11.8156 - val_loss: 0.0238 - val_mae: 0.0238 - val_mape: 6.8717
Epoch 42/60
22/27 [=======================>......] - ETA: 0s - loss: 0.0258 - mae: 0.0258 - mape: 11.9752
Epoch 42: val_loss did not improve from 0.02379
27/27 [==============================] - 1s 31ms/step - loss: 0.0252 - mae: 0.0252 - mape: 11.4341 - val_loss: 0.0313 - val_mae: 0.0313 - val_mape: 9.5432
Epoch 43/60
23/27 [========================>.....] - ETA: 0s - loss: 0.0240 - mae: 0.0240 - mape: 10.8324
Epoch 43: val_loss did not improve from 0.02379
27/27 [==============================] - 1s 25ms/step - loss: 0.0234 - mae: 0.0234 - mape: 10.6334 - val_loss: 0.0365 - val_mae: 0.0365 - val_mape: 11.8459
Epoch 44/60
26/27 [===========================>..] - ETA: 0s - loss: 0.0245 - mae: 0.0245 - mape: 11.3030
Epoch 44: val_loss did not improve from 0.02379
27/27 [==============================] - 1s 27ms/step - loss: 0.0245 - m

In [22]:

model = load_model(r'C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\E1-cp-0035-loss0.03.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 305ms/step
Mean Absolute Error (MAE): 3213.71
Median Absolute Error (MedAE): 2444.52
Mean Squared Error (MSE): 13784247.63
Root Mean Squared Error (RMSE): 3712.71
Mean Absolute Percentage Error (MAPE): 20.63 %
Median Absolute Percentage Error (MDAPE): 15.8 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# Fine Tuning

In [23]:
checkpoints = r'C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11'
model=r'C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\E1-cp-0035-loss0.03.h5'
start_epoch= 56

In [24]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\E1-cp-0035-loss0.03.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


In [25]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0213 - mae: 0.0213 - mape: 9.0499
Epoch 1: val_loss improved from inf to 0.02778, saving model to C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11
INFO:tensorflow:Assets written to: C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\assets


INFO:tensorflow:Assets written to: C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\assets


27/27 [==============================] - 6s 176ms/step - loss: 0.0212 - mae: 0.0212 - mape: 8.8196 - val_loss: 0.0278 - val_mae: 0.0278 - val_mape: 8.5849
Epoch 2/10
20/27 [=====================>........] - ETA: 0s - loss: 0.0204 - mae: 0.0204 - mape: 8.6105
Epoch 2: val_loss improved from 0.02778 to 0.02661, saving model to C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11
INFO:tensorflow:Assets written to: C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\assets


INFO:tensorflow:Assets written to: C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\assets


27/27 [==============================] - 2s 89ms/step - loss: 0.0207 - mae: 0.0207 - mape: 8.8208 - val_loss: 0.0266 - val_mae: 0.0266 - val_mape: 8.1438
Epoch 3/10
19/27 [====================>.........] - ETA: 0s - loss: 0.0200 - mae: 0.0200 - mape: 8.6232
Epoch 3: val_loss did not improve from 0.02661
27/27 [==============================] - 1s 22ms/step - loss: 0.0203 - mae: 0.0203 - mape: 8.5391 - val_loss: 0.0285 - val_mae: 0.0285 - val_mape: 8.7068
Epoch 4/10
23/27 [========================>.....] - ETA: 0s - loss: 0.0206 - mae: 0.0206 - mape: 8.3664
Epoch 4: val_loss improved from 0.02661 to 0.02627, saving model to C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11
INFO:tensorflow:Assets written to: C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\assets


INFO:tensorflow:Assets written to: C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\assets


27/27 [==============================] - 4s 163ms/step - loss: 0.0201 - mae: 0.0201 - mape: 8.0948 - val_loss: 0.0263 - val_mae: 0.0263 - val_mape: 8.1386
Epoch 5/10
26/27 [===========================>..] - ETA: 0s - loss: 0.0200 - mae: 0.0200 - mape: 8.0785
Epoch 5: val_loss did not improve from 0.02627
27/27 [==============================] - 1s 45ms/step - loss: 0.0200 - mae: 0.0200 - mape: 8.1533 - val_loss: 0.0296 - val_mae: 0.0296 - val_mape: 9.3040
Epoch 6/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0196 - mae: 0.0196 - mape: 8.5123
Epoch 6: val_loss did not improve from 0.02627
27/27 [==============================] - 1s 37ms/step - loss: 0.0201 - mae: 0.0201 - mape: 8.5956 - val_loss: 0.0274 - val_mae: 0.0274 - val_mape: 8.4544
Epoch 7/10
27/27 [==============================] - ETA: 0s - loss: 0.0196 - mae: 0.0196 - mape: 8.0652
Epoch 7: val_loss improved from 0.02627 to 0.02611, saving model to C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11
INFO:tensorflow:Asse

INFO:tensorflow:Assets written to: C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\assets


27/27 [==============================] - 3s 100ms/step - loss: 0.0196 - mae: 0.0196 - mape: 8.0652 - val_loss: 0.0261 - val_mae: 0.0261 - val_mape: 8.1859
Epoch 8/10
24/27 [=========================>....] - ETA: 0s - loss: 0.0200 - mae: 0.0200 - mape: 8.1784
Epoch 8: val_loss did not improve from 0.02611
27/27 [==============================] - 2s 68ms/step - loss: 0.0200 - mae: 0.0200 - mape: 8.5726 - val_loss: 0.0270 - val_mae: 0.0270 - val_mape: 8.2243
Epoch 9/10
23/27 [========================>.....] - ETA: 0s - loss: 0.0199 - mae: 0.0199 - mape: 8.0161
Epoch 9: val_loss did not improve from 0.02611
27/27 [==============================] - 2s 77ms/step - loss: 0.0199 - mae: 0.0199 - mape: 8.1337 - val_loss: 0.0280 - val_mae: 0.0280 - val_mape: 8.9451
Epoch 10/10
20/27 [=====================>........] - ETA: 0s - loss: 0.0194 - mae: 0.0194 - mape: 7.5663
Epoch 10: val_loss did not improve from 0.02611
27/27 [==============================] - 1s 49ms/step - loss: 0.0198 - mae: 0.0198

In [26]:

model = load_model(r'C:\Users\Muhib Ur-Rahman\Desktop\lab_10_11\E1-cp-0035-loss0.03.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 1s 1s/step
Mean Absolute Error (MAE): 3213.71
Median Absolute Error (MedAE): 2444.52
Mean Squared Error (MSE): 13784247.63
Root Mean Squared Error (RMSE): 3712.71
Mean Absolute Percentage Error (MAPE): 20.63 %
Median Absolute Percentage Error (MDAPE): 15.8 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)
